# Water Quality Surrogate Model -- Low-level Interface

This example demonstrates how to use the water quality surrogate model proposed in *MeGA-MP: Metric Graph Advection Message Passing*, Janine Strotherm, Luca Hermes, André Artelt, Barbara Hammer, Transactions of Machine Learning Research (2026), https://openreview.net/forum?id=2aZPFrKYYb.
Note that this surrogate model supports single species scenarios only.

In this example, we show how to use the low-level interface for setting-up and using the surrogate model predicting chlorine (CL2) concentrations at every node in the Hanoi network based on a setpoint source at the reservoir.

Note that there also exist a high-level interface for easier usage if no full control over all parameters of the surrogate is needed.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a target=\"_blank\" href=\"https://colab.research.google.com/github/WaterFutures/EPyT-Control/blob/main/docs/examples/quality_surrogate_ex.ipynb\"><img src=\"https://colab.research.google.com/assets/colab-badge.svg\" alt=\"Open In Colab\"/></a>'))

Because of the large depenecies, the surrogate model is not included by default but must be explicitly installed by either installing all optional dependencies (`[all]`) or only this surrogate model (`[qualsurrogate]`):

In [ ]:
%pip install epyt-control[qualsurrogate] --quiet

In [ ]:
import numpy as np
from epyt_control.models import QualitySurrogateModelEx, make_edge_index, get_edge_attribute, flow_to_velocity
from epyt_flow.utils import to_seconds, plot_timeseries_prediction, plot_timeseries_data
from epyt_flow.data.networks import load_hanoi

from sim_utils import run_simulation

This example is based on Hanoi.

In [ ]:
f_inp_in = load_hanoi().f_inp_in
f_msx_in = "simplecl2.msx"  # No topological information included! Can be used for any network!
sources_at = ['1']  # Chlorine source at the reservoir

In [ ]:
N_SECONDS = to_seconds(hours=12)
HYDRAULIC_STEP = 60
QUALITY_TIMESTEP = 1
nsteps = int(N_SECONDS / HYDRAULIC_STEP)
times = np.linspace(0, N_SECONDS, nsteps)

Create a random wave pattern to model the concentration of CL2 at the source node.

In [ ]:
def generate_injection_pattern(num_points, min_modes=3, max_modes=30, seed=None):
    rs = np.random.RandomState(seed)
    
    t = np.linspace(0, 2 * np.pi, num_points)
    signal = np.zeros(num_points)
    num_modes = rs.randint(min_modes, max_modes)
    decay_power = rs.uniform(0.5, 1.5)
    
    for k in range(1, num_modes + 1):
        amplitude_scale = 1.0 / (k**decay_power)
        a_k = rs.normal() * amplitude_scale
        b_k = rs.normal() * amplitude_scale
        signal += a_k * np.cos(k * t) + b_k * np.sin(k * t)
        
    return (signal - signal.min()) / (signal.max() - signal.min())

In [ ]:
injection_pattern = generate_injection_pattern((N_SECONDS // HYDRAULIC_STEP) + 1, seed=42)

plot_timeseries_data(injection_pattern.reshape(1, -1), x_axis_label="Time steps", y_axis_label="CL2 concentration at source node")

In [ ]:
sim_kwargs = {
    'duration' : N_SECONDS,
    'hydraulic_step' : HYDRAULIC_STEP,
    'quality_step' : QUALITY_TIMESTEP,
    'f_msx_in' : f_msx_in,
    'source_conc' : {n_id: injection_pattern for n_id in sources_at}
}

scada_data = run_simulation(f_inp_in, **sim_kwargs)  # Uses EPyT-Flow for running EPANET and EPANET-MSX simulations

# Extrac ground truth concentration
if sim_kwargs["f_msx_in"] is None:
    conc = scada_data.get_data_nodes_quality()
else:
    conc = scada_data.get_data_bulk_species_node_concentration()

#### Build water quality surrogate model

Because we want to have full control over all parameters, we create a new instance of [epyt_control.models.QualitySurrogateModelEx](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.qualsurrogate_mega_mp.quality_surrogate.QualitySurrogateModelEx) and pass several arguments to the constructur:

In [ ]:
# Create parameters for the surrogate model
topo = scada_data.network_topo
initial_state = np.zeros(topo.get_number_of_nodes())    # Zero concentration everywhere in the beginning

output_times = [nsteps * HYDRAULIC_STEP]
edge_index = make_edge_index(topo)
edge_lengths = get_edge_attribute(topo, 'length')
edge_diameters = get_edge_attribute(topo, 'diameter')/10/100  # Make sure to get the units right! -- i.e., mm to meters

# Create surrogate model
surrogate = QualitySurrogateModelEx(edge_index=edge_index,
                                    interpolation='bilinear',
                                    max_msg_passing_rounds=1000)

We can now use the surrogate to predict the concentration of our single species (here, Chlorine) at all nodes, based on one or multiple sources -- the concentration at those sources is set to a given pattern.

For this, we can either use the [QualitySurrogateModelEx.predict_ex()](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.qualsurrogate_mega_mp.quality_surrogate.QualitySurrogateModelEx.predict_ex) function of `QualitySurrogateModelEx` which provides us with full control over the prediction process and computes several quantities of interst as an output.
Alternatively, we can use the [QualitySurrogateModelEx.predict_conc()](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.qualsurrogate_mega_mp.quality_surrogate.QualitySurrogateModelEx.predict_conc) function if we are interest in the node-wise concentration over time only.

In [ ]:
# Build boundary conditions based on the source pattern
boundary_indices =  [topo.get_all_nodes().index(n) for n in sim_kwargs["source_conc"].keys()]
boundary_values = np.concatenate((np.array([0]), injection_pattern[:-1])).reshape(1, -1)
#boundary_values = conc[:, boundary_indices].T  # Alternatively, we could use the results form the ground truth simulation

# Build flow field
edge_flows = scada_data.get_data_flows()
flow_field = flow_to_velocity(topo, edge_flows).T

pred, edge_passes, _, agg_time, aggs_all = surrogate.predict_ex(initial_state=initial_state,     # Initial concentation at all nodes
                                                                control_indices=boundary_indices,  # Indices of source nodes
                                                                control_inputs=boundary_values,  # Concentation of those source nodes over time
                                                                flow_field=flow_field,           # Flow velocity at all links
                                                                edge_lengths=edge_lengths,       # Length of all links in m
                                                                edge_diameters=edge_diameters,   # Diameter of all links in m
                                                                output_times=output_times,
                                                                nsteps=nsteps,
                                                                dt=HYDRAULIC_STEP)

# Alternatively, you can call .predict_conc if you are interest in the node-wise concentration over time
conc_pred = surrogate.predict_conc(initial_state=initial_state,
                                   control_indices=boundary_indices,
                                   control_inputs=boundary_values,
                                   flow_field=flow_field,
                                   edge_lengths=edge_lengths,
                                   edge_diameters=edge_diameters,
                                   output_times=output_times,
                                   nsteps=nsteps,
                                   dt=HYDRAULIC_STEP)
print(conc_pred.shape)  # 1. dimension: time; 2. dimension: nodes

#### Evaluation

In [ ]:
print(f"MAE: {np.mean(np.abs(conc_pred - conc))}")

In [ ]:
# Plot ground truth concentration vs. prediction from the surrogate model at six random nodes
for idx in np.random.RandomState(32).choice(range(scada_data.network_topo.get_number_of_nodes()), size=6):
    plot_timeseries_prediction(conc[:, idx], conc_pred[:, idx],
                               x_axis_label="Time steps", y_axis_label="CL2 concentration")